# Notebook 06: Computer Vision - Optical Character Recognition (OCR)

## Learning Objectives

In this notebook, you will learn:
- How to extract text from images using OCR
- How TrOCR (Transformer-based OCR) works for printed and handwritten text
- How to use PaddleOCR for multi-language, full-page OCR
- How to evaluate OCR accuracy on real datasets
- When to use TrOCR vs PaddleOCR for different tasks

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | microsoft/trocr-small-printed | 558MB | 8GB | 8GB RAM, CPU | English printed text |
| **Large (GPU)** | microsoft/trocr-large-printed | ~1.5GB | 12GB | 10GB VRAM | Higher accuracy |
| **PaddleOCR** | PaddleOCR | ~3.5GB | 8GB | GPU recommended | 80+ languages, full-page |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `PIL`
- Optional: `paddlepaddle`, `paddleocr` for PaddleOCR section

## Expected Behaviors

**First Time Running**:
- TrOCR-small download: ~558MB (3-5 minutes)
- Downloads both a vision encoder (image understanding) and text decoder (text generation)
- Cached in `~/.cache/huggingface/hub/` for subsequent runs

**OCR Output**:
- TrOCR returns a single text string per image crop
- Works best on **cropped single-line text regions**, not full pages
- PaddleOCR returns bounding boxes + text for full pages

**Performance**:
- CPU: 2-5 seconds per image (TrOCR), 5-15 seconds (PaddleOCR)
- GPU: <1 second per image (TrOCR), 1-3 seconds (PaddleOCR)

**Accuracy**:
- Printed text: 90-98% character accuracy
- Handwritten text: 70-90% (depends on legibility)
- Quality degrades with noise, rotation, or low resolution

## Overview

**OCR (Optical Character Recognition)** extracts machine-readable text from images.

**Use Cases**: Document digitization, receipt processing, license plate recognition, form extraction, accessibility tools.

### TrOCR Architecture

TrOCR is an encoder-decoder transformer:
1. **Vision Encoder** (ViT): Converts image patches into embeddings
2. **Text Decoder** (RoBERTa-based): Auto-regressively generates text tokens

This is the same sequence-to-sequence pattern used in image captioning and machine translation, but specialized for text recognition.

### TrOCR vs PaddleOCR

| Feature | TrOCR | PaddleOCR |
|---------|-------|-----------|
| Input | Single-line text crops | Full pages |
| Detection | No (needs pre-cropped text) | Yes (finds text regions) |
| Languages | English (default models) | 80+ languages |
| Architecture | Transformer encoder-decoder | Detection + Recognition pipeline |
| Best for | High-accuracy single-line OCR | Production full-page OCR |

## Setup and Installation

In [ ]:
import sys, time, random, os, warnings
from io import BytesIO

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, set_seed
from PIL import Image, ImageDraw, ImageFont
import requests
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")

## Model Selection

In [ ]:
# Option 1: Small model (CPU-friendly, English printed text)
MODEL_NAME = "microsoft/trocr-small-printed"  # 558MB

# Option 2: Large model (GPU-optimized, higher accuracy)
# MODEL_NAME = "microsoft/trocr-large-printed"  # ~1.5GB

# Option 3: Handwritten text (if working with handwriting)
# MODEL_NAME = "microsoft/trocr-small-handwritten"  # 558MB

print(f"Selected model: {MODEL_NAME}")

## Method 1: TrOCR for Single-Line Text

TrOCR works best on cropped images containing a single line of text. We load the processor (handles image preprocessing) and the model (encoder-decoder transformer).

In [ ]:
try:
    print(f"Loading {MODEL_NAME}...")
    processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
    model.to(device)
    print(f"Model loaded on {device}.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Ensure you have internet access and sufficient disk space (~558MB).")

In [ ]:
def extract_text(image: Image.Image) -> str:
    """Extract text from a single-line text image using TrOCR.

    Args:
        image: PIL Image containing a single line of text.

    Returns:
        Extracted text string.
    """
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

### Basic Text Extraction

Let's test TrOCR on a handwritten text sample from the IAM dataset.

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL.

    Args:
        url: HTTP(S) URL pointing to an image file.

    Returns:
        PIL Image in RGB mode.
    """
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")


# Test with handwritten text sample
url = "https://fki.tic.heia-fr.ch/static/img/a01-122-02.jpg"
try:
    image = load_image_from_url(url)
    print(f"Image size: {image.size}")

    text = extract_text(image)
    print(f"Extracted text: {text}")

    # Display image
    plt.figure(figsize=(10, 3))
    plt.imshow(image)
    plt.title(f"Extracted: {text}", fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not load sample image: {e}")
    print("Continuing with synthetic test images below.")

### Processing Multiple Text Regions

In practice, you often need to extract text from multiple cropped regions of a document.

In [ ]:
def ocr_multiple_images(image_urls: list[str]) -> pd.DataFrame:
    """Extract text from multiple images.

    Args:
        image_urls: List of URLs pointing to text images.

    Returns:
        DataFrame with image index, extracted text, and processing time.
    """
    rows = []
    for i, url in enumerate(image_urls, 1):
        try:
            img = load_image_from_url(url)
            start = time.perf_counter()
            text = extract_text(img)
            elapsed = time.perf_counter() - start
            rows.append({"#": i, "Text": text, "Time (s)": f"{elapsed:.2f}"})
        except Exception as e:
            rows.append({"#": i, "Text": f"Error: {e}", "Time (s)": "N/A"})

    return pd.DataFrame(rows)


test_urls = [
    "https://fki.tic.heia-fr.ch/static/img/a01-122-02.jpg",
    "https://fki.tic.heia-fr.ch/static/img/a01-122-02-00.jpg",
]

print("Multi-Image OCR")
ocr_multiple_images(test_urls)

### Synthetic Text Test

We can create synthetic images with known text to objectively measure OCR accuracy.

In [ ]:
def create_text_image(
    text: str, width: int = 400, height: int = 60, font_size: int = 24
) -> Image.Image:
    """Create a synthetic image containing text.

    Args:
        text: Text to render in the image.
        width: Image width in pixels.
        height: Image height in pixels.
        font_size: Font size for the text.

    Returns:
        PIL Image with rendered text.
    """
    img = Image.new("RGB", (width, height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except (OSError, IOError):
        font = ImageFont.load_default()
    draw.text((10, 10), text, fill=(0, 0, 0), font=font)
    return img


def evaluate_ocr_accuracy(test_cases: list[str]) -> pd.DataFrame:
    """Evaluate TrOCR accuracy on synthetic text images.

    Args:
        test_cases: List of ground truth text strings.

    Returns:
        DataFrame with ground truth, predicted text, and character accuracy.
    """
    rows = []
    for gt_text in test_cases:
        img = create_text_image(gt_text)
        predicted = extract_text(img)

        # Character-level accuracy
        matches = sum(a == b for a, b in zip(gt_text, predicted))
        max_len = max(len(gt_text), len(predicted), 1)
        accuracy = matches / max_len

        rows.append({
            "Ground Truth": gt_text,
            "Predicted": predicted,
            "Char Accuracy": f"{accuracy:.2%}",
        })

    return pd.DataFrame(rows)


test_texts = [
    "Hello World",
    "The quick brown fox",
    "12345 67890",
    "invoice@company.com",
    "$1,234.56 USD",
]

print("Synthetic Text OCR Accuracy")
evaluate_ocr_accuracy(test_texts)

## Method 2: Evaluation on MNIST Digits

Let's test TrOCR on handwritten digits from the MNIST dataset. Since TrOCR is trained on full text lines (not individual digits), accuracy on single digits may vary.

In [ ]:
def evaluate_on_mnist(num_samples: int = 20) -> pd.DataFrame:
    """Evaluate TrOCR on MNIST handwritten digits.

    Args:
        num_samples: Number of MNIST samples to evaluate.

    Returns:
        DataFrame with digit, prediction, and correctness.
    """
    try:
        import torchvision.datasets as datasets

        print("Loading MNIST test dataset...")
        mnist = datasets.MNIST(root="./data", train=False, download=True)
        print(f"Dataset loaded: {len(mnist)} images")

        rows = []
        correct = 0
        for i in range(min(num_samples, len(mnist))):
            img, label = mnist[i]
            img_rgb = img.convert("RGB").resize((128, 128))
            predicted = extract_text(img_rgb).strip()
            is_correct = predicted == str(label)
            if is_correct:
                correct += 1
            rows.append({
                "Digit": label,
                "Predicted": predicted,
                "Correct": is_correct,
            })

        df = pd.DataFrame(rows)
        print(f"\nAccuracy: {correct}/{num_samples} ({100*correct/num_samples:.1f}%)")
        print("Note: TrOCR is trained on text lines, not individual digits.")
        return df

    except Exception as e:
        print(f"Could not load MNIST: {e}")
        return pd.DataFrame()


evaluate_on_mnist(20)

## Method 3: PaddleOCR for Full-Page OCR (Optional)

PaddleOCR is a production-grade OCR system that handles full pages with text detection, recognition, and layout analysis. It supports 80+ languages. This section requires the `paddleocr` package.

In [ ]:
# Uncomment to install PaddleOCR:
# !pip install paddlepaddle paddleocr

try:
    from paddleocr import PaddleOCR
    import cv2

    PADDLE_AVAILABLE = True
    print("PaddleOCR available.")

    ocr = PaddleOCR(lang="en", use_gpu=torch.cuda.is_available())
    print("PaddleOCR initialized (English).")

except ImportError:
    PADDLE_AVAILABLE = False
    print("PaddleOCR not installed. Skipping full-page OCR section.")
    print("Install with: pip install paddlepaddle paddleocr")

In [ ]:
def paddle_ocr_image(image_path_or_url: str) -> pd.DataFrame:
    """Run PaddleOCR on an image and return detected text regions.

    Args:
        image_path_or_url: Local path or URL to an image.

    Returns:
        DataFrame with detected text, confidence, and bounding box.
    """
    if not PADDLE_AVAILABLE:
        print("PaddleOCR not available.")
        return pd.DataFrame()

    # Load image
    if image_path_or_url.startswith("http"):
        img = load_image_from_url(image_path_or_url)
        img_array = np.array(img)
    else:
        img_array = cv2.imread(image_path_or_url)

    result = ocr.ocr(img_array, cls=True)

    rows = []
    if result and result[0]:
        for line in result[0]:
            bbox = line[0]
            text = line[1][0]
            confidence = line[1][1]
            rows.append({
                "Text": text,
                "Confidence": f"{confidence:.3f}",
                "Top-Left": f"({bbox[0][0]:.0f}, {bbox[0][1]:.0f})",
            })

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Text", "Confidence", "Top-Left"])


if PADDLE_AVAILABLE:
    print("PaddleOCR Full-Page Detection")
    # Create a synthetic document image for testing
    doc_img = Image.new("RGB", (600, 200), (255, 255, 255))
    draw = ImageDraw.Draw(doc_img)
    try:
        font = ImageFont.truetype("arial.ttf", 20)
    except (OSError, IOError):
        font = ImageFont.load_default()
    draw.text((20, 20), "Invoice #12345", fill=(0, 0, 0), font=font)
    draw.text((20, 60), "Total: $1,234.56", fill=(0, 0, 0), font=font)
    draw.text((20, 100), "Date: 2024-01-15", fill=(0, 0, 0), font=font)
    doc_img.save("temp_test_doc.png")

    result_df = paddle_ocr_image("temp_test_doc.png")
    display(result_df)

    os.remove("temp_test_doc.png")
else:
    print("Skipped: PaddleOCR not installed.")

In [ ]:
# Visualize PaddleOCR detections with bounding boxes
if PADDLE_AVAILABLE:
    import matplotlib.patches as patches

    doc_img = Image.new("RGB", (600, 200), (255, 255, 255))
    draw = ImageDraw.Draw(doc_img)
    try:
        font = ImageFont.truetype("arial.ttf", 20)
    except (OSError, IOError):
        font = ImageFont.load_default()
    draw.text((20, 20), "Invoice #12345", fill=(0, 0, 0), font=font)
    draw.text((20, 60), "Total: $1,234.56", fill=(0, 0, 0), font=font)
    doc_img.save("temp_vis_doc.png")

    result = ocr.ocr(np.array(doc_img), cls=True)

    fig, ax = plt.subplots(1, figsize=(12, 5))
    ax.imshow(doc_img)

    if result and result[0]:
        for line in result[0]:
            bbox = line[0]
            text = line[1][0]
            x_min = min(p[0] for p in bbox)
            y_min = min(p[1] for p in bbox)
            width = max(p[0] for p in bbox) - x_min
            height = max(p[1] for p in bbox) - y_min
            rect = patches.Rectangle((x_min, y_min), width, height,
                                     linewidth=2, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
            ax.text(x_min, y_min - 5, text, color="red", fontsize=9)

    ax.set_title("PaddleOCR: Detected Text Regions", fontsize=14, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    os.remove("temp_vis_doc.png")
else:
    print("Skipped: PaddleOCR not installed.")

### Multi-Language OCR

PaddleOCR supports 80+ languages. You initialize with a different `lang` parameter for each language.

In [ ]:
language_df = pd.DataFrame([
    {"Language": "English", "Code": "en", "Script": "Latin"},
    {"Language": "Chinese", "Code": "ch", "Script": "CJK"},
    {"Language": "Japanese", "Code": "japan", "Script": "CJK + Kana"},
    {"Language": "Korean", "Code": "korean", "Script": "Hangul"},
    {"Language": "French", "Code": "fr", "Script": "Latin"},
    {"Language": "German", "Code": "german", "Script": "Latin"},
    {"Language": "Arabic", "Code": "ar", "Script": "Arabic"},
    {"Language": "Hindi", "Code": "hi", "Script": "Devanagari"},
])

print("PaddleOCR Supported Languages (sample)")
print("Full list: https://github.com/PaddlePaddle/PaddleOCR/blob/release/2.7/doc/doc_en/multi_languages_en.md\n")
language_df

## Practical Applications

### Application 1: Receipt/Invoice Processing

In [ ]:
def process_receipt_lines(texts: list[str]) -> pd.DataFrame:
    """Extract text from multiple receipt line images.

    Args:
        texts: List of text strings to render and OCR.

    Returns:
        DataFrame with original text, OCR result, and match status.
    """
    rows = []
    for text in texts:
        img = create_text_image(text, width=500, height=50)
        predicted = extract_text(img)
        rows.append({
            "Original": text,
            "OCR Result": predicted,
            "Match": predicted.strip() == text.strip(),
        })
    return pd.DataFrame(rows)


receipt_lines = [
    "Coffee          $4.50",
    "Sandwich       $12.99",
    "Tax             $1.40",
    "Total          $18.89",
]

print("Receipt OCR")
process_receipt_lines(receipt_lines)

### Application 2: Batch Document Processing

In [ ]:
def batch_ocr(texts: list[str]) -> pd.DataFrame:
    """Batch-process multiple text images and measure throughput.

    Args:
        texts: List of text strings to render and OCR.

    Returns:
        DataFrame with timing statistics.
    """
    start = time.perf_counter()
    results = []
    for text in texts:
        img = create_text_image(text)
        predicted = extract_text(img)
        results.append(predicted)
    elapsed = time.perf_counter() - start

    return pd.DataFrame([
        {"Metric": "Images processed", "Value": len(texts)},
        {"Metric": "Total time (s)", "Value": f"{elapsed:.2f}"},
        {"Metric": "Avg per image (s)", "Value": f"{elapsed / len(texts):.2f}"},
        {"Metric": "Throughput (images/s)", "Value": f"{len(texts) / elapsed:.1f}"},
    ])


batch_texts = [f"Document line number {i}" for i in range(1, 11)]
print("Batch Processing Throughput")
batch_ocr(batch_texts)

## Performance Benchmarking

In [ ]:
def benchmark_ocr(n_runs: int = 10) -> pd.DataFrame:
    """Benchmark TrOCR inference speed.

    Args:
        n_runs: Number of runs to average over.

    Returns:
        DataFrame with timing statistics.
    """
    img = create_text_image("Hello World")

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        extract_text(img)
        times.append(time.perf_counter() - start)

    return pd.DataFrame([
        {"Metric": "Model", "Value": MODEL_NAME.split('/')[-1]},
        {"Metric": "Device", "Value": str(device)},
        {"Metric": "Mean (ms)", "Value": f"{np.mean(times) * 1000:.1f}"},
        {"Metric": "Std (ms)", "Value": f"{np.std(times) * 1000:.1f}"},
        {"Metric": "Min (ms)", "Value": f"{np.min(times) * 1000:.1f}"},
        {"Metric": "Max (ms)", "Value": f"{np.max(times) * 1000:.1f}"},
    ])


print("TrOCR Inference Benchmark")
benchmark_ocr()

### TrOCR vs PaddleOCR Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {"Aspect": "Model Size", "TrOCR": "558MB (small)", "PaddleOCR": "~3.5GB"},
    {"Aspect": "Input Type", "TrOCR": "Single-line crops", "PaddleOCR": "Full pages"},
    {"Aspect": "Text Detection", "TrOCR": "No", "PaddleOCR": "Yes (built-in)"},
    {"Aspect": "Languages", "TrOCR": "English (default)", "PaddleOCR": "80+"},
    {"Aspect": "Architecture", "TrOCR": "Transformer enc-dec", "PaddleOCR": "Det + Rec pipeline"},
    {"Aspect": "Speed (CPU)", "TrOCR": "2-5s per crop", "PaddleOCR": "5-15s per page"},
    {"Aspect": "Best For", "TrOCR": "Single-line accuracy", "PaddleOCR": "Production full-page"},
])

print("TrOCR vs PaddleOCR")
comparison_df

## Error Analysis

Let's test TrOCR on challenging inputs to understand where it struggles.

In [ ]:
def analyze_failure_modes() -> pd.DataFrame:
    """Test OCR on challenging synthetic inputs.

    Returns:
        DataFrame with test case, ground truth, prediction, and notes.
    """
    test_cases = [
        ("Normal text", "Hello World"),
        ("Numbers only", "9876543210"),
        ("Special chars", "!@#$%^&*()"),
        ("Mixed case", "aBcDeFgHiJ"),
        ("Long text", "The quick brown fox jumps over the lazy dog near the riverbank"),
        ("Very short", "Hi"),
        ("Email format", "user@example.com"),
        ("URL format", "https://example.com/path"),
    ]

    rows = []
    for category, gt in test_cases:
        img = create_text_image(gt, width=600)
        predicted = extract_text(img)
        exact_match = predicted.strip() == gt.strip()
        rows.append({
            "Category": category,
            "Ground Truth": gt[:40],
            "Predicted": predicted[:40],
            "Exact Match": exact_match,
        })

    return pd.DataFrame(rows)


print("Error Analysis: Challenging Inputs")
analyze_failure_modes()

**Observations**: TrOCR handles standard printed text well but may struggle with special characters, URLs, and email formats that were underrepresented in training data. Very short inputs may also produce unexpected outputs since the model expects longer text sequences.

## State-of-the-Art OCR Models

In [ ]:
sota_df = pd.DataFrame([
    {"Model": "TrOCR", "Size": "558MB-1.5GB", "Type": "Single-line", "Languages": "English", "Best For": "High-accuracy text recognition"},
    {"Model": "PaddleOCR", "Size": "~3.5GB", "Type": "Full-page", "Languages": "80+", "Best For": "Production multilingual OCR"},
    {"Model": "EasyOCR", "Size": "~1GB", "Type": "Full-page", "Languages": "80+", "Best For": "Easy setup, good accuracy"},
    {"Model": "Tesseract", "Size": "~50MB", "Type": "Full-page", "Languages": "100+", "Best For": "Lightweight, open-source classic"},
    {"Model": "GOT-OCR", "Size": "~2GB", "Type": "Full-page", "Languages": "Multi", "Best For": "Document understanding + OCR"},
])

print("OCR Model Landscape")
sota_df

## Exercises

1. **Custom Images**: Create images with different fonts, sizes, and backgrounds. How does accuracy change?

2. **Handwritten vs Printed**: Compare `trocr-small-printed` vs `trocr-small-handwritten` on the same images.

3. **Document Processing**: Build a pipeline that takes a full document image, crops text regions manually, and extracts text from each.

4. **Noise Robustness**: Add Gaussian noise or blur to synthetic images and measure accuracy degradation.

5. **Multi-Language**: If PaddleOCR is installed, test with Chinese, Japanese, or Arabic text images.

## Key Takeaways

- **TrOCR** uses a transformer encoder-decoder for high-accuracy single-line OCR
- Separate TrOCR models exist for **printed** vs **handwritten** text
- TrOCR works best on **cropped single-line text regions**, not full pages
- **PaddleOCR** handles full-page OCR with built-in text detection and 80+ languages
- OCR accuracy depends on image quality, font, and text complexity
- For production systems, combine text detection (find text regions) with recognition (read text)

## Next Steps & Resources

**Next Notebook**: Notebook 07 -- Speech Recognition with Whisper

**Documentation**:
- [TrOCR Paper](https://arxiv.org/abs/2109.10282)
- [HuggingFace TrOCR Guide](https://huggingface.co/docs/transformers/model_doc/trocr)
- [PaddleOCR GitHub](https://github.com/PaddlePaddle/PaddleOCR)
- [IAM Handwriting Database](https://fki.tic.heia-fr.ch/databases/iam-handwriting-database)